# Capítulo 2 — NumPy II: broadcasting, agregaciones, máscaras y aleatoriedad

**Serie Ciencia de Datos · Tomo 2 — Herramientas Python para Data Science**  
José Fernando Meza

Código del capítulo, extraído del libro. Repositorio: https://github.com/josefer76/serie-ciencia-de-datos

> Ejecute las celdas en orden: algunas dependen de variables definidas antes.

## Código del capítulo

### 2.1 Broadcasting: reglas formales de compatibilidad de formas y errores frecuentes

In [ ]:
import numpy as np

matriz = np.array([[1, 2, 3], [4, 5, 6]])       # forma (2, 3)
vector_fila = np.array([10, 20, 30])            # forma (3,)

print(matriz + vector_fila)
# [[11 22 33]
#  [14 25 36]]
# El vector se "repite" conceptualmente en cada una de las 2 filas.

In [ ]:
col = np.array([[100], [200]])   # forma (2, 1)
print(matriz + col)
# [[101 102 103]
#  [204 205 206]]
# (2, 3) y (2, 1): el eje de columnas (1 frente a 3) se expande -> (2, 3).
# Esto produce, en la práctica, una "suma exterior" fila por fila.

In [ ]:
vector_a = np.array([1, 2, 3])   # forma (3,)
vector_b = np.array([10, 20])    # forma (2,)
try:
    vector_a + vector_b
except ValueError as error:
    print(f"ValueError: {error}")
# Ninguna dimensión vale 1 y 3 != 2: no hay forma de conciliarlas.

*Salida en el libro:*

```text
ValueError: operands could not be broadcast together with shapes (3,) (2,)
```

### 2.2 Funciones universales (ufuncs) y agregaciones con el parámetro `axis`

In [ ]:
import numpy as np

datos = np.array([[1, 2, 3], [4, 5, 6]])   # forma (2, 3)

print(datos.sum())
# 21          -> suma de los 6 elementos, sin eje
print(datos.sum(axis=0))
# [5 7 9]     -> colapsa el eje 0 (filas): una suma POR COLUMNA
print(datos.sum(axis=1))
# [ 6 15]     -> colapsa el eje 1 (columnas): una suma POR FILA
print(datos.mean(axis=0))     # [2.5 3.5 4.5]

In [ ]:
print(datos.sum(axis=1, keepdims=True))
# [[ 6]
#  [15]]
print(datos.sum(axis=1, keepdims=True).shape)   # (2, 1), no (2,)

### 2.3 Indexación booleana y máscaras: filtrar sin condicionales explícitos; `np.where()`

In [ ]:
import numpy as np

valores = np.array([5, -3, 12, 0, -8, 20])
mascara = valores > 0
print(mascara)                  # [ True False  True False False  True]
print(valores[mascara])
# [ 5 12 20]   -> solo los elementos donde la máscara es True

In [ ]:
try:
    bool(np.array([True, False]))
except ValueError as error:
    print(error)

*Salida en el libro:*

```text
The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
```

In [ ]:
print(valores[(valores > 0) & (valores < 15)])   # [ 5 12]

In [ ]:
print(np.where(valores > 0, valores, 0))
# [ 5  0 12  0  0 20]   -> conserva los positivos, reemplaza el resto por 0

In [ ]:
print(np.where(valores > 0))     # (array([0, 2, 5]),)

### 2.4 Reestructuración: `reshape()`, `ravel()`, `transpose()`, `concatenate()`, `stack()`

In [ ]:
import numpy as np

base = np.arange(6)              # array([0, 1, 2, 3, 4, 5])
print(base.reshape(2, 3))
# [[0 1 2]
#  [3 4 5]]

# -1 le indica a NumPy que infiera esa dimensión a partir de las demás
print(base.reshape(2, -1))       # equivalente a reshape(2, 3)
# [[0 1 2]
#  [3 4 5]]

In [ ]:
m = np.array([[1, 2], [3, 4]])
vista = m.ravel()
vista[0] = 99
print(m)          # el original cambió: ravel() fue una vista
# [[99  2]
#  [ 3  4]]

m2 = np.array([[1, 2], [3, 4]])
copia = m2.flatten()
copia[0] = 99
print(m2)         # el original NO cambió: flatten() fue una copia
# [[1 2]
#  [3 4]]

In [ ]:
print(m2.T)
# [[1 3]
#  [2 4]]

In [ ]:
a1 = np.array([[1, 2], [3, 4]])   # forma (2, 2)
a2 = np.array([[5, 6]])           # forma (1, 2)
print(np.concatenate([a1, a2], axis=0))
# [[1 2]
#  [3 4]
#  [5 6]]
# -> forma (3, 2): se unieron a lo largo de un eje YA EXISTENTE

s1 = np.array([1, 2, 3])          # forma (3,)
s2 = np.array([4, 5, 6])          # forma (3,)
print(np.stack([s1, s2]))
# [[1 2 3]
#  [4 5 6]]
# -> forma (2, 3): se creó un eje NUEVO (ninguno de los originales tenía 2
#    filas)

print(np.concatenate([s1, s2]))
# [1 2 3 4 5 6], forma (6,) -> mismo eje, sin crear dimensión

### 2.5 Álgebra lineal aplicada: producto escalar, operador `@`, y `np.linalg.norm()`

In [ ]:
import numpy as np

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print(A * B)
# [[ 5 12]
#  [21 32]]
# -> Hadamard: cada posición multiplicada por su correspondiente. NO es
#    álgebra lineal.

print(A @ B)
# [[19 22]
#  [43 50]]
# -> producto matricial real: fila-por-columna, como en álgebra lineal.

In [ ]:
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])

print(u.dot(v))    # 32   ->  1*4 + 2*5 + 3*6
print(u @ v)
# 32   ->  forma equivalente y preferida desde Python 3.5 (PEP 465)

In [ ]:
print(np.linalg.norm(u))    # 3.7416573867739413  -> raíz(1^2 + 2^2 + 3^2)

matriz_normas = np.array([[3, 4], [6, 8]])
print(np.linalg.norm(matriz_normas, axis=1))   # norma de CADA FILA
# [ 5. 10.]
print(np.linalg.norm(matriz_normas, axis=0))   # norma de CADA COLUMNA
# [6.70820393 8.94427191]

In [ ]:
def similitud_coseno(a, b):
    """Similitud coseno entre dos vectores 1D, en el rango [-1, 1]."""
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

perfil_x = np.array([5, 0, 3, 0, 2])
perfil_y = np.array([4, 0, 0, 0, 1])
print(similitud_coseno(perfil_x, perfil_y))   # 0.8655784229010971

In [ ]:
def similitud_coseno_segura(a, b):
    """Similitud coseno con guarda explícita contra vectores nulos."""
    norma_a = np.linalg.norm(a)
    norma_b = np.linalg.norm(b)
    if norma_a == 0 or norma_b == 0:
        raise ValueError("La similitud coseno no está definida para el "
                         "vector cero.")
    return (a @ b) / (norma_a * norma_b)

### 2.6 Números aleatorios con `default_rng()`: la semilla y la reproducibilidad como exigencia científica

In [ ]:
import numpy as np

semilla = 42
rng = np.random.default_rng(semilla)   # objeto Generador con semilla fija

# 3 flotantes uniformes en [0, 1)
print(rng.random(3))
# [0.77395605 0.43887844 0.85859792]

# 5 enteros aleatorios en [0, 10)
print(rng.integers(0, 10, size=5))
# [0 6 2 0 5]

# 3 muestras de una normal estándar
print(rng.normal(loc=0, scale=1, size=3))
# [ 0.1278404  -0.31624259 -0.01680116]

In [ ]:
rng_a = np.random.default_rng(42)
rng_b = np.random.default_rng(42)
print(np.array_equal(rng_a.random(3), rng_b.random(3)))   # True

### 2.7 Tipos de datos, promoción implícita y casting: qué ocurre al operar enteros con flotantes

In [ ]:
import numpy as np

a_entero = np.array([1, 2, 3], dtype=np.int64)
b_flotante = np.array([1.5, 2.5, 3.5], dtype=np.float64)
print((a_entero + b_flotante).dtype)
# float64: el entero se promueve a flotante

In [ ]:
c_int32 = np.array([1, 2, 3], dtype=np.int32)
print((c_int32 + 5).dtype)
# int32   -> escalar entero débil: NO fuerza upcast
print((c_int32 + 5.0).dtype)
# float64 -> escalar flotante: SÍ cambia de categoría

In [ ]:
d_int8 = np.array([100], dtype=np.int8)
print(d_int8 + 50)
# [-106]   -> 150 no cabe en int8 ([-128, 127]); se desborda en silencio

In [ ]:
e = np.array([3.9, -2.7, 2.5])
print(e.astype(np.int64))
# [ 3 -2  2]   -> trunca, no redondea (3.9 -> 3, no 4)
print(np.round(e).astype(np.int64))    # [ 4 -3  2]   -> redondeo previo

### 2.8 Valores faltantes: `np.nan`, su naturaleza contagiosa y las funciones `nan*`

In [ ]:
import numpy as np

datos = np.array([1.0, 2.0, np.nan, 4.0])
print(datos.sum())     # nan  -> un solo nan invalida la suma completa
print(datos.mean())    # nan  -> lo mismo para cualquier agregación estándar

In [ ]:
print(np.nan == np.nan)   # False

In [ ]:
print(np.isnan(datos))    # [False False  True False]

In [ ]:
try:
    np.array([1, 2, np.nan], dtype=np.int64)
except ValueError as error:
    print(error)   # cannot convert float NaN to integer

In [ ]:
print(np.nansum(datos))    # 7.0                 -> ignora el nan
print(np.nanmean(datos))
# 2.3333333333333335   -> media de los 3 valores válidos

## Soluciones de los ejercicios

### [Ex-L2-C2-01] Normalización z-score por columnas con broadcasting

#### Desarrollo con código

In [ ]:
import numpy as np

generador = np.random.default_rng(2024)
FILAS = 8

# Tres variables en escalas deliberadamente muy distintas
matriz_datos = np.column_stack([
    generador.normal(45, 12, FILAS),        # edad:       decenas
    generador.normal(35000, 9000, FILAS),   # ingreso:    decenas de miles
    generador.random(FILAS),                # puntuación: entre 0 y 1
])

print("medias originales por columna:", matriz_datos.mean(axis=0).round(2))
print("desviaciones por columna     :", matriz_datos.std(axis=0).round(2))

# ---------------------------------------------
# NORMALIZACIÓN: una sola expresión, sin bucles
# ---------------------------------------------
# axis=0 COLAPSA las filas -> un estadístico POR COLUMNA (Sección 2.2)
medias_por_columna = matriz_datos.mean(axis=0)          # forma (3,)
desviaciones_por_columna = matriz_datos.std(axis=0)     # forma (3,)

# Broadcasting: (8, 3) con (3,) -> la forma (3,) se lee como (1, 3)
# y se difunde sobre las 8 filas sin copiar memoria (Sección 2.1).
matriz_normalizada = ((matriz_datos - medias_por_columna)
                      / desviaciones_por_columna)
print("forma de matriz_datos      :", matriz_datos.shape)
print("forma de medias_por_columna:", medias_por_columna.shape)
print("forma del resultado        :", matriz_normalizada.shape)

# ---------------------------------------------
# VERIFICACIÓN: por construcción, media 0 y desviación 1 por columna
# ---------------------------------------------
print("media por columna:", matriz_normalizada.mean(axis=0).round(10))
print("std por columna  :", matriz_normalizada.std(axis=0).round(10))

#### Salida esperada

*Salida en el libro:*

```text
medias originales por columna: [4.933000e+01 3.917002e+04 5.600000e-01]
desviaciones por columna     : [1.19400e+01 8.51383e+03 2.70000e-01]
forma de matriz_datos      : (8, 3)
forma de medias_por_columna: (3,)
forma del resultado        : (8, 3)
media por columna: [-0. -0.  0.]
std por columna  : [1. 1. 1.]
```

In [ ]:
mal_normalizada = (
    (matriz_datos - matriz_datos.mean(axis=1, keepdims=True))
    / matriz_datos.std(axis=1, keepdims=True))

print(mal_normalizada.mean(axis=1).round(10))
# [ 0. -0.  0.  0.  0.  0.  0.  0.]  <- "verificado"
print(mal_normalizada.mean(axis=0).round(3))
# [-0.706  1.414 -0.708]      <- pero NO por columna

### [Ex-L2-C2-02] Detección y tratamiento de valores atípicos con máscaras

#### Desarrollo con código

In [ ]:
import numpy as np

generador = np.random.default_rng(7)

# Mediciones de un sensor: normal centrada en 20.0 con desviación 1.5
mediciones_sensor = generador.normal(20.0, 1.5, 200)

# Se inyectan tres anomalías en posiciones conocidas, para poder verificar
posiciones_anomalas = [15, 87, 143]
mediciones_sensor[posiciones_anomalas] = [45.0, -12.0, 38.5]

# ---------------------------------------------
# 1. CARACTERIZAR LA DISTRIBUCIÓN
# ---------------------------------------------
media_muestral = mediciones_sensor.mean()
desviacion_muestral = mediciones_sensor.std()
UMBRAL_SIGMAS = 3

# ---------------------------------------------
# 2. IDENTIFICAR CON UNA MÁSCARA BOOLEANA
# ---------------------------------------------
# ufunc vectorizada
distancia_a_la_media = np.abs(mediciones_sensor - media_muestral)
mascara_atipicos = distancia_a_la_media > UMBRAL_SIGMAS * desviacion_muestral

print(f"atípicos detectados: {mascara_atipicos.sum()} de "
      f"{len(mediciones_sensor)}")
# np.where con UN argumento: devuelve las posiciones
print("posiciones:", np.where(mascara_atipicos)[0])
print("valores   :", mediciones_sensor[mascara_atipicos].round(2))

# ---------------------------------------------
# 3. CORREGIR SIN TOCAR EL ORIGINAL
# ---------------------------------------------
# np.where con TRES argumentos: condicional ternario vectorizado.
# Devuelve un array NUEVO; 'mediciones_sensor' queda intacto.
mediciones_corregidas = np.where(mascara_atipicos, media_muestral,
                                 mediciones_sensor)

print(f"media: {media_muestral:.3f} -> {mediciones_corregidas.mean():.3f}")
print(f"std  : {desviacion_muestral:.3f} -> "
      f"{mediciones_corregidas.std():.3f}")

#### Salida esperada

*Salida en el libro:*

```text
atípicos detectados: 3 de 200
posiciones: [ 15  87 143]
valores   : [ 45.  -12.   38.5]
media: 19.864 -> 19.804
std  : 3.417 -> 1.303
```

In [ ]:
# Estadísticos calculados SIN los atípicos detectados
sin_atipicos = mediciones_sensor[~mascara_atipicos]
print(f"media SIN los atípicos: {sin_atipicos.mean():.3f}")
print(f"std SIN los atípicos  : {sin_atipicos.std():.3f}"
      f"  -> umbral 3s = ±{3 * sin_atipicos.std():.2f}")
print(f"std CON los atípicos  : {desviacion_muestral:.3f}"
      f"  -> umbral 3s = ±{3 * desviacion_muestral:.2f}")
print(f"el umbral se ensanchó {desviacion_muestral / sin_atipicos.std():.1f} "
      f"veces")

# Alternativa robusta: mediana y desviación absoluta mediana (MAD)
mediana = np.median(mediciones_sensor)
mad = np.median(np.abs(mediciones_sensor - mediana))
mascara_robusta = np.abs(mediciones_sensor - mediana) > 3 * 1.4826 * mad
print(f"MAD escalado          : {1.4826 * mad:.3f}")
print(f"atípicos según el MAD : {mascara_robusta.sum()}")

*Salida en el libro:*

```text
media SIN los atípicos: 19.803
std SIN los atípicos  : 1.313  -> umbral 3s = ±3.94
std CON los atípicos  : 3.417  -> umbral 3s = ±10.25
el umbral se ensanchó 2.6 veces
MAD escalado          : 1.278
atípicos según el MAD : 3
```

### [Ex-L2-C2-03] Similitud coseno vectorizada frente al Índice de Jaccard

#### Desarrollo con código

In [ ]:
import numpy as np

idiomas = {
    "Español":   {"casa", "sol", "luna", "agua", "noche", "mano", "libro",
                  "flor", "mar", "luz"},
    "Inglés":    {"house", "sun", "moon", "water", "night", "hand", "book",
                  "flower", "sea", "light"},
    "Portugués": {"casa", "sol", "lua", "agua", "noite", "mao", "livro",
                  "flor", "mar", "luz"},
    "Francés":   {"maison", "soleil", "lune", "eau", "nuit", "main", "livre",
                  "fleur", "mer", "lumiere"},
}
nombres_idiomas = list(idiomas.keys())

# ---------------------------------------------
# 1. VECTORIZACIÓN: de conjuntos a coordenadas numéricas
# ---------------------------------------------
# El vocabulario conjunto define los EJES del espacio vectorial.
# Se ordena para que la representación sea reproducible.
vocabulario_conjunto = sorted(set().union(*idiomas.values()))

# Cada idioma pasa a ser un vector binario: 1 si contiene la palabra, 0 si no.
matriz_binaria = np.array([
    [1 if palabra in idiomas[idioma] else 0
     for palabra in vocabulario_conjunto]
    for idioma in nombres_idiomas
])
print(matriz_binaria.shape)
# (4, 34) -> 4 idiomas en un espacio de 34 dimensiones

# ---------------------------------------------
# 2. SIMILITUD COSENO PARA TODOS LOS PARES DE UNA VEZ
# ---------------------------------------------
# Truco vectorizado: si cada fila se normaliza a longitud 1, el producto
# matricial de la matriz por su transpuesta da TODAS las similitudes coseno.
# forma (4, 1)
normas = np.linalg.norm(matriz_binaria, axis=1, keepdims=True)
# broadcasting
matriz_unitaria = matriz_binaria / normas
# forma (4, 4)
matriz_coseno = matriz_unitaria @ matriz_unitaria.T

# ---------------------------------------------
# 3. ÍNDICE DE JACCARD, PARA COMPARAR
# ---------------------------------------------
def indice_jaccard(conjunto_a, conjunto_b):
    """J(A, B) = |A ∩ B| / |A ∪ B|."""
    union = len(conjunto_a | conjunto_b)
    return len(conjunto_a & conjunto_b) / union if union else 1.0

matriz_jaccard = np.array([
    [indice_jaccard(idiomas[a], idiomas[b]) for b in nombres_idiomas]
    for a in nombres_idiomas
])

# ---------------------------------------------
# 4. RESULTADOS
# ---------------------------------------------
print("\nMatriz de similitud coseno:")
print(" " * 10 + "".join(f"{nombre:>10}" for nombre in nombres_idiomas))
for nombre, fila in zip(nombres_idiomas, matriz_coseno):
    print(f"{nombre:<10}" + "".join(f"{valor:>10.3f}" for valor in fila))

print("\nComparación de ambas métricas, por pares:")
print(f"{'Par':<20}{'Jaccard':>9}{'Coseno':>9}")
for i in range(len(nombres_idiomas)):
    for j in range(i + 1, len(nombres_idiomas)):
        par = f"{nombres_idiomas[i]}-{nombres_idiomas[j]}"
        print(f"{par:<20}{matriz_jaccard[i, j]:>9.3f}"
              f"{matriz_coseno[i, j]:>9.3f}")

#### Salida esperada

*Salida en el libro:*

```text
(4, 34)

Matriz de similitud coseno:
             Español    Inglés Portugués   Francés
Español        1.000     0.000     0.600     0.000
Inglés         0.000     1.000     0.000     0.000
Portugués      0.600     0.000     1.000     0.000
Francés        0.000     0.000     0.000     1.000

Comparación de ambas métricas, por pares:
Par                   Jaccard   Coseno
Español-Inglés          0.000    0.000
Español-Portugués       0.429    0.600
Español-Francés         0.000    0.000
Inglés-Portugués        0.000    0.000
Inglés-Francés          0.000    0.000
Portugués-Francés       0.000    0.000
```